# 15. The Automated Guided Vehicle Traffic Management Problem

## Tier 3 — Metaheuristic Optimization (Genetic Algorithm & Particle Swarm)

This notebook explores **advanced metaheuristic algorithms** that balance solution quality and computational efficiency for larger AGV traffic management problems.

### Learning goals

- Understand how **population-based metaheuristics** explore complex solution spaces
- See how **evolutionary operators** can improve AGV scheduling over time
- Learn how **swarm intelligence** can coordinate multiple AGVs effectively

### What this notebook outputs

- **Genetic Algorithm** with crossover, mutation, and selection
- **Particle Swarm Optimization** for AGV path coordination
- **Hybrid approach** combining multiple metaheuristics
- Performance comparison with previous tiers

### Why metaheuristics matter

Metaheuristics provide **near-optimal solutions** for complex problems where exact methods are too slow and simple heuristics are too basic, making them ideal for medium-to-large terminal operations.

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from itertools import product, combinations
    from dataclasses import dataclass, field
    from typing import List, Tuple, Dict, Optional, Set, Callable
    from collections import defaultdict, deque
    import heapq
    import time
    import random
    import copy
    import math
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Large-scale instance (12 AGVs, 16 nodes)

For Tier-3, we use an even larger instance to demonstrate metaheuristic capabilities:

- **16 nodes** (4x4 grid terminal layout)
- **48 directed edges** (dense road network)
- **12 AGVs** (significant coordination challenge)
- **Time horizon** of 40 time units

### Enhanced complexity factors

- **Multiple path options** between origins and destinations
- **Varying AGV speeds** and capabilities
- **Dynamic congestion** effects on travel times

In [ ]:
# ----------------------------
# Enhanced data models for metaheuristics
# ----------------------------
@dataclass(frozen=True)
class Node:
    id: int
    x: float
    y: float
    is_intersection: bool = True
    capacity: int = 1
    congestion_factor: float = 1.0  # Dynamic congestion multiplier

@dataclass(frozen=True)
class Edge:
    from_node: int
    to_node: int
    base_travel_time: int
    capacity: int = 1
    bidirectional: bool = False
    congestion_sensitive: bool = True

@dataclass(frozen=True)
class AGV:
    id: int
    origin: int
    destination: int
    priority: int = 1
    max_speed: float = 1.0
    size: int = 1  # AGV size for collision detection
    battery_capacity: float = 100.0  # For extended modeling

@dataclass
class AGVPlan:
    agv_id: int
    path: List[int]
    schedule: List[int]  # Time indices for each node
    total_time: int
    conflicts: int = 0
    fitness: float = 0.0

# ----------------------------
# Large terminal layout: 16 nodes in a 4x4 grid
# ----------------------------
# Layout:
#   1  --  2  --  3  --  4
#   |      |      |      |
#   5  --  6  --  7  --  8
#   |      |      |      |
#   9  -- 10  -- 11  -- 12
#   |      |      |      |
#  13  -- 14  -- 15  -- 16
#
nodes = [
    Node(1, 0.0, 3.0), Node(2, 1.5, 3.0), Node(3, 3.0, 3.0), Node(4, 4.5, 3.0),
    Node(5, 0.0, 2.0), Node(6, 1.5, 2.0), Node(7, 3.0, 2.0), Node(8, 4.5, 2.0),
    Node(9, 0.0, 1.0), Node(10, 1.5, 1.0), Node(11, 3.0, 1.0), Node(12, 4.5, 1.0),
    Node(13, 0.0, 0.0), Node(14, 1.5, 0.0), Node(15, 3.0, 0.0), Node(16, 4.5, 0.0)
]

# Dense road network with varying travel times
edges = [
    # Horizontal edges (all rows)
    Edge(1, 2, 2), Edge(2, 1, 2), Edge(2, 3, 2), Edge(3, 2, 2),
    Edge(3, 4, 2), Edge(4, 3, 2),
    Edge(5, 6, 1), Edge(6, 5, 1), Edge(6, 7, 1), Edge(7, 6, 1),
    Edge(7, 8, 1), Edge(8, 7, 1),
    Edge(9, 10, 1), Edge(10, 9, 1), Edge(10, 11, 1), Edge(11, 10, 1),
    Edge(11, 12, 1), Edge(12, 11, 1),
    Edge(13, 14, 1), Edge(14, 13, 1), Edge(14, 15, 1), Edge(15, 14, 1),
    Edge(15, 16, 1), Edge(16, 15, 1),
    
    # Vertical edges (all columns)
    Edge(1, 5, 1), Edge(5, 1, 1), Edge(2, 6, 1), Edge(6, 2, 1),
    Edge(3, 7, 1), Edge(7, 3, 1), Edge(4, 8, 2), Edge(8, 4, 2),
    Edge(5, 9, 1), Edge(9, 5, 1), Edge(6, 10, 1), Edge(10, 6, 1),
    Edge(7, 11, 1), Edge(11, 7, 1), Edge(8, 12, 2), Edge(12, 8, 2),
    Edge(9, 13, 1), Edge(13, 9, 1), Edge(10, 14, 1), Edge(14, 10, 1),
    Edge(11, 15, 1), Edge(15, 11, 1), Edge(12, 16, 2), Edge(16, 12, 2),
    
    # Diagonal shortcuts (for more routing options)
    Edge(1, 6, 3), Edge(6, 1, 3), Edge(2, 7, 3), Edge(7, 2, 3),
    Edge(3, 8, 3), Edge(8, 3, 3), Edge(5, 10, 3), Edge(10, 5, 3),
    Edge(6, 11, 3), Edge(11, 6, 3), Edge(7, 12, 3), Edge(12, 7, 3),
    Edge(9, 14, 3), Edge(14, 9, 3), Edge(10, 15, 3), Edge(15, 10, 3),
    Edge(11, 16, 3), Edge(16, 11, 3)
]

# More AGVs with diverse characteristics
agvs = [
    AGV(1, 1, 16, 1, 1.2, 1, 100.0),
    AGV(2, 4, 13, 2, 1.0, 1, 95.0),
    AGV(3, 2, 15, 3, 0.8, 1, 90.0),
    AGV(4, 5, 12, 2, 1.1, 1, 85.0),
    AGV(5, 10, 7, 1, 1.3, 1, 80.0),
    AGV(6, 7, 10, 3, 0.9, 1, 75.0),
    AGV(7, 12, 5, 2, 1.0, 1, 70.0),
    AGV(8, 9, 8, 1, 1.4, 1, 65.0),
    AGV(9, 14, 3, 2, 1.0, 1, 60.0),
    AGV(10, 11, 6, 3, 0.7, 1, 55.0),
    AGV(11, 16, 1, 1, 1.1, 1, 50.0),
    AGV(12, 13, 4, 2, 1.2, 1, 45.0)
]

# ----------------------------
# Problem parameters
# ----------------------------
TIME_HORIZON = 40
SAFETY_HEADWAY = 1
NODE_CAPACITY = 1

# ----------------------------
# Helper data structures
# ----------------------------
node_lookup = {n.id: n for n in nodes}
edge_lookup = {(e.from_node, e.to_node): e for e in edges}
agv_lookup = {a.id: a for a in agvs}

outgoing_edges = {n.id: [e for e in edges if e.from_node == n.id] for n in nodes}
incoming_edges = {n.id: [e for e in edges if e.to_node == n.id] for n in nodes}

print(f"Large terminal: {len(nodes)} nodes, {len(edges)} edges, {len(agvs)} AGVs")
print(f"Time horizon: {TIME_HORIZON} units")
print(f"AGV speed range: {min(a.max_speed for a in agvs):.1f} - {max(a.max_speed for a in agvs):.1f}")

## Step 1 — Advanced path planning with multiple options

For metaheuristics, we need **multiple path alternatives** for each AGV to enable solution diversity:

In [ ]:
class AdvancedPathPlanner:
    """Advanced path planner with multiple routing options"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge]):
        self.nodes = nodes
        self.edges = edges
        self.node_lookup = {n.id: n for n in nodes}
        self.edge_lookup = {(e.from_node, e.to_node): e for e in edges}
        self.adjacency = self._build_adjacency()
        self.path_cache = {}  # Cache for repeated path calculations
    
    def _build_adjacency(self) -> Dict[int, List[Tuple[int, int]]]:
        """Build adjacency list with congestion consideration"""
        adj = defaultdict(list)
        for edge in self.edges:
            travel_time = edge.base_travel_time
            if edge.congestion_sensitive:
                travel_time = int(travel_time * self.node_lookup[edge.to_node].congestion_factor)
            adj[edge.from_node].append((edge.to_node, travel_time))
        return dict(adj)
    
    def dijkstra_with_congestion(self, start: int, end: int, 
                               congestion_map: Dict[int, float] = None) -> Tuple[List[int], int]:
        """Dijkstra with dynamic congestion consideration"""
        if congestion_map:
            # Update congestion factors temporarily
            for node_id, factor in congestion_map.items():
                if node_id in self.node_lookup:
                    self.node_lookup[node_id].congestion_factor = factor
            self.adjacency = self._build_adjacency()
        
        distances = {node.id: float('inf') for node in self.nodes}
        distances[start] = 0
        previous = {}
        pq = [(0, start)]
        visited = set()
    
        while pq:
            current_dist, current_node = heapq.heappop(pq)
            
            if current_node in visited:
                continue
            
            visited.add(current_node)
            
            if current_node == end:
                break
            
            for neighbor, travel_time in self.adjacency.get(current_node, []):
                if neighbor in visited:
                    continue
                
                distance = current_dist + travel_time
                
                if distance < distances[neighbor]:
                    distances[neighbor] = distance
                    previous[neighbor] = current_node
                    heapq.heappush(pq, (distance, neighbor))
        
        # Reconstruct path
        if end not in previous and start != end:
            return [], float('inf')
        
        path = []
        current = end
        while current is not None:
            path.append(current)
            current = previous.get(current)
        
        path.reverse()
        return path, distances[end]
    
    def find_diverse_paths(self, start: int, end: int, k: int = 5, 
                          congestion_map: Dict[int, float] = None) -> List[Tuple[List[int], int]]:
        """Find k diverse paths using edge removal and perturbation"""
        cache_key = (start, end, k, tuple(sorted(congestion_map.items())) if congestion_map else ())
        if cache_key in self.path_cache:
            return self.path_cache[cache_key]
        
        paths = []
        removed_edges = set()
        
        for attempt in range(k):
            # Temporarily remove edges from previous paths
            temp_edges = self.edges.copy()
            self.edges = [e for e in self.edges if (e.from_node, e.to_node) not in removed_edges]
            self.adjacency = self._build_adjacency()
            
            # Find path with current edge set
            path, cost = self.dijkstra_with_congestion(start, end, congestion_map)
            
            if path and path not in [p[0] for p in paths]:
                paths.append((path, cost))
                
                # Add some edges to removal set for diversity
                if len(path) > 1:
                    for i in range(len(path) - 1):
                        if random.random() < 0.3:  # 30% chance to remove edge
                            removed_edges.add((path[i], path[i + 1]))
            
            # Restore edges
            self.edges = temp_edges
            self.adjacency = self._build_adjacency()
            
            if len(paths) >= k:
                break
        
        # Sort by cost
        paths.sort(key=lambda x: x[1])
        
        # Cache result
        self.path_cache[cache_key] = paths
        return paths
    
    def calculate_path_travel_time(self, path: List[int], agv_speed: float = 1.0) -> int:
        """Calculate actual travel time for a path considering AGV speed"""
        if len(path) < 2:
            return 0
        
        total_time = 0
        for i in range(len(path) - 1):
            from_node, to_node = path[i], path[i + 1]
            edge_key = (from_node, to_node)
            
            if edge_key in self.edge_lookup:
                base_time = self.edge_lookup[edge_key].base_travel_time
            else:
                base_time = 1
            
            # Adjust for AGV speed
            adjusted_time = base_time / agv_speed
            total_time += int(adjusted_time)
        
        return total_time

# Initialize advanced path planner
advanced_planner = AdvancedPathPlanner(nodes, edges)

# Test diverse path finding
print("=== DIVERSE PATH FINDING TEST ===")
test_agv = agvs[0]  # AGV 1: 1 → 16
paths = advanced_planner.find_diverse_paths(test_agv.origin, test_agv.destination, k=5)

print(f"Found {len(paths)} diverse paths for AGV {test_agv.id}:")
for i, (path, cost) in enumerate(paths):
    print(f"  Path {i+1}: {path} (cost: {cost})")

## Step 2 — Genetic Algorithm for AGV scheduling

The GA evolves a population of AGV schedules using **genetic operators**:

- **Chromosome encoding**: Path and schedule representation
- **Crossover**: Exchange routing strategies between solutions
- **Mutation**: Introduce random changes for diversity
- **Selection**: Choose best solutions for reproduction

In [ ]:
class GeneticAlgorithmScheduler:
    """Genetic Algorithm for AGV traffic management"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge], agvs: List[AGV], 
                 path_planner: AdvancedPathPlanner):
        self.nodes = nodes
        self.edges = edges
        self.agvs = agvs
        self.path_planner = path_planner
        self.node_lookup = {n.id: n for n in nodes}
        
        # GA parameters
        self.population_size = 50
        self.generations = 100
        self.crossover_rate = 0.8
        self.mutation_rate = 0.2
        self.elite_size = 5
        
        # Pre-compute path options for each AGV
        self.path_options = {}
        for agv in agvs:
            self.path_options[agv.id] = path_planner.find_diverse_paths(
                agv.origin, agv.destination, k=5
            )
    
    def create_chromosome(self) -> Dict[int, AGVPlan]:
        """Create a random chromosome (individual solution)"""
        chromosome = {}
        
        for agv in self.agvs:
            # Randomly select a path from available options
            path_options = self.path_options[agv.id]
            if path_options:
                path, _ = random.choice(path_options)
                
                # Create schedule with some random delay
                start_delay = random.randint(0, 5)
                travel_time = self.path_planner.calculate_path_travel_time(path, agv.max_speed)
                
                schedule = list(range(start_delay, start_delay + len(path)))
                total_time = start_delay + travel_time
                
                plan = AGVPlan(
                    agv_id=agv.id,
                    path=path,
                    schedule=schedule,
                    total_time=total_time
                )
                
                chromosome[agv.id] = plan
        
        return chromosome
    
    def calculate_fitness(self, chromosome: Dict[int, AGVPlan]) -> float:
        """Calculate fitness function (lower is better for minimization)"""
        if not chromosome:
            return float('inf')
        
        # Calculate total completion time weighted by priority
        total_weighted_time = 0
        conflicts = 0
        
        for agv_id, plan in chromosome.items():
            agv = self.agv_lookup[agv_id]
            weighted_time = plan.total_time * agv.priority
            total_weighted_time += weighted_time
        
        # Calculate conflicts
        conflicts = self.count_conflicts(chromosome)
        
        # Fitness is weighted combination of time and conflicts
        fitness = total_weighted_time + (conflicts * 10)  # Penalize conflicts heavily
        
        return fitness
    
    def count_conflicts(self, chromosome: Dict[int, AGVPlan]) -> int:
        """Count number of conflicts in a solution"""
        conflicts = 0
        
        # Check node conflicts
        node_usage = defaultdict(list)  # (time, node) -> [agv_ids]
        
        for agv_id, plan in chromosome.items():
            for i, node in enumerate(plan.path):
                if i < len(plan.schedule):
                    time = plan.schedule[i]
                    node_usage[(time, node)].append(agv_id)
        
        for (time, node), agv_list in node_usage.items():
            if len(agv_list) > 1:
                conflicts += len(agv_list) - 1
        
        # Check edge conflicts (simplified)
        edge_usage = defaultdict(list)  # (time, from, to) -> [agv_ids]
        
        for agv_id, plan in chromosome.items():
            for i in range(len(plan.path) - 1):
                if i < len(plan.schedule):
                    time = plan.schedule[i]
                    from_node = plan.path[i]
                    to_node = plan.path[i + 1]
                    edge_usage[(time, from_node, to_node)].append(agv_id)
        
        for (time, from_node, to_node), agv_list in edge_usage.items():
            if len(agv_list) > 1:
                conflicts += len(agv_list) - 1
        
        return conflicts
    
    def tournament_selection(self, population: List[Dict[int, AGVPlan]], 
                           tournament_size: int = 3) -> Dict[int, AGVPlan]:
        """Tournament selection for genetic operators"""
        tournament = random.sample(population, min(tournament_size, len(population)))
        
        # Return the best individual (lowest fitness)
        best = min(tournament, key=self.calculate_fitness)
        return best
    
    def crossover(self, parent1: Dict[int, AGVPlan], parent2: Dict[int, AGVPlan]) -> Dict[int, AGVPlan]:
        """Crossover operator to combine two parents"""
        if random.random() > self.crossover_rate:
            return random.choice([parent1, parent2])
        
        child = {}
        
        # Single-point crossover on AGV plans
        crossover_point = random.randint(1, len(self.agvs) - 1)
        agv_ids = [agv.id for agv in self.agvs]
        
        for i, agv_id in enumerate(agv_ids):
            if i < crossover_point:
                child[agv_id] = copy.deepcopy(parent1[agv_id]) if agv_id in parent1 else None
            else:
                child[agv_id] = copy.deepcopy(parent2[agv_id]) if agv_id in parent2 else None
        
        return child
    
    def mutate(self, chromosome: Dict[int, AGVPlan]) -> Dict[int, AGVPlan]:
        """Mutation operator to introduce diversity"""
        mutated = copy.deepcopy(chromosome)
        
        for agv_id, plan in mutated.items():
            if random.random() < self.mutation_rate:
                # Mutate by changing path or schedule
                mutation_type = random.choice(['path', 'schedule', 'delay'])
                
                if mutation_type == 'path' and len(self.path_options[agv_id]) > 1:
                    # Change to a different path
                    new_path, _ = random.choice(self.path_options[agv_id])
                    plan.path = new_path
                    plan.schedule = list(range(plan.schedule[0], plan.schedule[0] + len(new_path)))
                    
                elif mutation_type == 'schedule':
                    # Shift entire schedule
                    shift = random.randint(-2, 2)
                    plan.schedule = [max(0, t + shift) for t in plan.schedule]
                    
                elif mutation_type == 'delay':
                    # Add or remove initial delay
                    delay_change = random.randint(-1, 2)
                    new_delay = max(0, plan.schedule[0] + delay_change)
                    plan.schedule = list(range(new_delay, new_delay + len(plan.path)))
                    plan.total_time = new_delay + self.path_planner.calculate_path_travel_time(
                        plan.path, self.agv_lookup[agv_id].max_speed
                    )
        
        return mutated
    
    def evolve_population(self, population: List[Dict[int, AGVPlan]]) -> List[Dict[int, AGVPlan]]:
        """Evolve one generation of the population"""
        # Calculate fitness for all individuals
        fitness_scores = [(self.calculate_fitness(ind), ind) for ind in population]
        fitness_scores.sort(key=lambda x: x[0])  # Sort by fitness (lower is better)
        
        # Keep elite individuals
        new_population = [ind for fitness, ind in fitness_scores[:self.elite_size]]
        
        # Generate offspring
        while len(new_population) < self.population_size:
            parent1 = self.tournament_selection(population)
            parent2 = self.tournament_selection(population)
            
            child = self.crossover(parent1, parent2)
            child = self.mutate(child)
            
            new_population.append(child)
        
        return new_population
    
    def run_ga(self) -> Tuple[Dict[int, AGVPlan], List[float]]:
        """Run the complete genetic algorithm"""
        print("=== GENETIC ALGORITHM OPTIMIZATION ===")
        
        # Initialize population
        population = [self.create_chromosome() for _ in range(self.population_size)]
        
        best_fitness_history = []
        best_individual = None
        best_fitness = float('inf')
        
        # Evolution loop
        for generation in range(self.generations):
            population = self.evolve_population(population)
            
            # Track best fitness
            current_best = min(population, key=self.calculate_fitness)
            current_fitness = self.calculate_fitness(current_best)
            
            if current_fitness < best_fitness:
                best_fitness = current_fitness
                best_individual = copy.deepcopy(current_best)
            
            best_fitness_history.append(best_fitness)
            
            if generation % 20 == 0:
                print(f"Generation {generation}: Best fitness = {best_fitness:.2f}")
        
        print(f"GA completed. Best fitness: {best_fitness:.2f}")
        
        return best_individual, best_fitness_history

# Initialize GA scheduler
ga_scheduler = GeneticAlgorithmScheduler(nodes, edges, agvs, advanced_planner)

print("Genetic Algorithm initialized:")
print(f"Population size: {ga_scheduler.population_size}")
print(f"Generations: {ga_scheduler.generations}")
print(f"Crossover rate: {ga_scheduler.crossover_rate}")
print(f"Mutation rate: {ga_scheduler.mutation_rate}")

## Step 3 — Particle Swarm Optimization

PSO uses **swarm intelligence** where particles explore the solution space and share information:

- **Particles**: Each represents a complete AGV schedule
- **Velocity**: Direction and magnitude of search
- **Personal best**: Best solution found by each particle
- **Global best**: Best solution found by the entire swarm

In [ ]:
class ParticleSwarmOptimizer:
    """Particle Swarm Optimization for AGV scheduling"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge], agvs: List[AGV],
                 path_planner: AdvancedPathPlanner):
        self.nodes = nodes
        self.edges = edges
        self.agvs = agvs
        self.path_planner = path_planner
        self.agv_lookup = {a.id: a for a in agvs}
        
        # PSO parameters
        self.swarm_size = 30
        self.max_iterations = 80
        self.w = 0.7  # Inertia weight
        self.c1 = 1.5  # Cognitive coefficient
        self.c2 = 1.5  # Social coefficient
        
        # Pre-compute path options
        self.path_options = {}
        for agv in agvs:
            self.path_options[agv.id] = path_planner.find_diverse_paths(
                agv.origin, agv.destination, k=5
            )
    
    class Particle:
        """Represents a particle in the swarm"""
        def __init__(self, position: Dict[int, AGVPlan]):
            self.position = position  # Current solution
            self.velocity = self._initialize_velocity(position)
            self.personal_best = copy.deepcopy(position)
            self.personal_best_fitness = float('inf')
        
        def _initialize_velocity(self, position: Dict[int, AGVPlan]) -> Dict[int, Dict]:
            """Initialize velocity for each AGV"""
            velocity = {}
            for agv_id, plan in position.items():
                velocity[agv_id] = {
                    'path_change': 0.0,
                    'schedule_shift': 0.0,
                    'delay_change': 0.0
                }
            return velocity
    
    def create_random_particle(self) -> Particle:
        """Create a random particle"""
        position = {}
        
        for agv in self.agvs:
            path_options = self.path_options[agv.id]
            if path_options:
                path, _ = random.choice(path_options)
                
                start_delay = random.randint(0, 8)
                travel_time = self.path_planner.calculate_path_travel_time(path, agv.max_speed)
                
                schedule = list(range(start_delay, start_delay + len(path)))
                total_time = start_delay + travel_time
                
                plan = AGVPlan(
                    agv_id=agv.id,
                    path=path,
                    schedule=schedule,
                    total_time=total_time
                )
                
                position[agv.id] = plan
        
        return self.Particle(position)
    
    def calculate_fitness(self, position: Dict[int, AGVPlan]) -> float:
        """Calculate fitness for a particle position"""
        if not position:
            return float('inf')
        
        total_weighted_time = 0
        conflicts = 0
        
        for agv_id, plan in position.items():
            agv = self.agv_lookup[agv_id]
            weighted_time = plan.total_time * agv.priority
            total_weighted_time += weighted_time
        
        # Count conflicts
        conflicts = self.count_conflicts(position)
        
        fitness = total_weighted_time + (conflicts * 15)  # Heavier penalty for PSO
        
        return fitness
    
    def count_conflicts(self, position: Dict[int, AGVPlan]) -> int:
        """Count conflicts in particle position"""
        conflicts = 0
        node_usage = defaultdict(list)
        
        for agv_id, plan in position.items():
            for i, node in enumerate(plan.path):
                if i < len(plan.schedule):
                    time = plan.schedule[i]
                    node_usage[(time, node)].append(agv_id)
        
        for (time, node), agv_list in node_usage.items():
            if len(agv_list) > 1:
                conflicts += len(agv_list) - 1
        
        return conflicts
    
    def update_velocity(self, particle: Particle, global_best: Dict[int, AGVPlan]):
        """Update particle velocity"""
        for agv_id in particle.position.keys():
            # Current velocity components
            v_path = particle.velocity[agv_id]['path_change']
            v_schedule = particle.velocity[agv_id]['schedule_shift']
            v_delay = particle.velocity[agv_id]['delay_change']
            
            # Cognitive component (personal best)
            if agv_id in particle.personal_best and agv_id in global_best:
                personal_path_diff = self._calculate_path_difference(
                    particle.position[agv_id], particle.personal_best[agv_id]
                )
                global_path_diff = self._calculate_path_difference(
                    particle.position[agv_id], global_best[agv_id]
                )
                
                # Update velocity with cognitive and social components
                v_path = (self.w * v_path + 
                         self.c1 * random.random() * personal_path_diff +
                         self.c2 * random.random() * global_path_diff)
                
                # Schedule and delay updates (simplified)
                schedule_diff = (particle.personal_best[agv_id].schedule[0] - 
                               particle.position[agv_id].schedule[0]) if particle.position[agv_id].schedule else 0
                global_schedule_diff = (global_best[agv_id].schedule[0] - 
                                      particle.position[agv_id].schedule[0]) if particle.position[agv_id].schedule else 0
                
                v_schedule = (self.w * v_schedule + 
                            self.c1 * random.random() * schedule_diff +
                            self.c2 * random.random() * global_schedule_diff)
                
                # Limit velocity
                v_path = max(-2.0, min(2.0, v_path))
                v_schedule = max(-3.0, min(3.0, v_schedule))
                v_delay = max(-2.0, min(2.0, v_delay))
            
            particle.velocity[agv_id] = {
                'path_change': v_path,
                'schedule_shift': v_schedule,
                'delay_change': v_delay
            }
    
    def _calculate_path_difference(self, current: AGVPlan, target: AGVPlan) -> float:
        """Calculate difference between two paths (simplified)"""
        if not current.path or not target.path:
            return 0.0
        
        # Simple difference based on path length and first node
        length_diff = len(target.path) - len(current.path)
        first_node_diff = (target.path[0] - current.path[0]) if current.path and target.path else 0
        
        return (length_diff * 0.1 + first_node_diff * 0.05)
    
    def update_position(self, particle: Particle):
        """Update particle position based on velocity"""
        for agv_id, plan in particle.position.items():
            velocity = particle.velocity[agv_id]
            
            # Update schedule based on velocity
            if plan.schedule:
                schedule_shift = int(velocity['schedule_shift'])
                new_start = max(0, plan.schedule[0] + schedule_shift)
                plan.schedule = list(range(new_start, new_start + len(plan.path)))
                plan.total_time = new_start + self.path_planner.calculate_path_travel_time(
                    plan.path, self.agv_lookup[agv_id].max_speed
                )
            
            # Path change (probabilistic)
            if random.random() < abs(velocity['path_change']) * 0.1:
                path_options = self.path_options[agv_id]
                if len(path_options) > 1:
                    new_path, _ = random.choice(path_options)
                    plan.path = new_path
                    plan.schedule = list(range(plan.schedule[0], plan.schedule[0] + len(new_path)))
    
    def run_pso(self) -> Tuple[Dict[int, AGVPlan], List[float]]:
        """Run Particle Swarm Optimization"""
        print("=== PARTICLE SWARM OPTIMIZATION ===")
        
        # Initialize swarm
        swarm = [self.create_random_particle() for _ in range(self.swarm_size)]
        
        # Initialize personal bests
        for particle in swarm:
            particle.personal_best_fitness = self.calculate_fitness(particle.position)
        
        # Initialize global best
        global_best = min(swarm, key=lambda p: p.personal_best_fitness)
        global_best_fitness = global_best.personal_best_fitness
        global_best_position = copy.deepcopy(global_best.position)
        
        fitness_history = []
        
        # PSO main loop
        for iteration in range(self.max_iterations):
            for particle in swarm:
                # Update velocity
                self.update_velocity(particle, global_best_position)
                
                # Update position
                self.update_position(particle)
                
                # Update personal best
                current_fitness = self.calculate_fitness(particle.position)
                if current_fitness < particle.personal_best_fitness:
                    particle.personal_best_fitness = current_fitness
                    particle.personal_best = copy.deepcopy(particle.position)
            
            # Update global best
            best_particle = min(swarm, key=lambda p: p.personal_best_fitness)
            if best_particle.personal_best_fitness < global_best_fitness:
                global_best_fitness = best_particle.personal_best_fitness
                global_best_position = copy.deepcopy(best_particle.personal_best)
            
            fitness_history.append(global_best_fitness)
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Best fitness = {global_best_fitness:.2f}")
        
        print(f"PSO completed. Best fitness: {global_best_fitness:.2f}")
        
        return global_best_position, fitness_history

# Initialize PSO optimizer
pso_optimizer = ParticleSwarmOptimizer(nodes, edges, agvs, advanced_planner)

print("Particle Swarm Optimizer initialized:")
print(f"Swarm size: {pso_optimizer.swarm_size}")
print(f"Max iterations: {pso_optimizer.max_iterations}")
print(f"Inertia weight: {pso_optimizer.w}")
print(f"Cognitive coefficient: {pso_optimizer.c1}")
print(f"Social coefficient: {pso_optimizer.c2}")

## Step 4 — Hybrid metaheuristic approach

Combine the strengths of GA and PSO in a **hybrid approach**:

- **GA exploration**: Broad search through genetic operators
- **PSO exploitation**: Focused search around promising solutions
- **Adaptive switching**: Choose the best method dynamically

In [ ]:
class HybridMetaheuristic:
    """Hybrid approach combining GA and PSO"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge], agvs: List[AGV],
                 path_planner: AdvancedPathPlanner):
        self.nodes = nodes
        self.edges = edges
        self.agvs = agvs
        self.path_planner = path_planner
        
        # Initialize both optimizers
        self.ga_scheduler = GeneticAlgorithmScheduler(nodes, edges, agvs, path_planner)
        self.pso_optimizer = ParticleSwarmOptimizer(nodes, edges, agvs, path_planner)
        
        # Hybrid parameters
        self.max_iterations = 60
        self.switch_frequency = 15  # Switch between GA and PSO every N iterations
    
    def run_hybrid(self) -> Tuple[Dict[int, AGVPlan], Dict[str, List[float]]]:
        """Run hybrid optimization"""
        print("=== HYBRID METAHEURISTIC OPTIMIZATION ===")
        print("Combining Genetic Algorithm and Particle Swarm Optimization")
        
        best_solution = None
        best_fitness = float('inf')
        
        ga_fitness_history = []
        pso_fitness_history = []
        hybrid_fitness_history = []
        
        # Start with GA for initial exploration
        print("\nPhase 1: Genetic Algorithm Exploration")
        ga_solution, ga_history = self.ga_scheduler.run_ga()
        ga_fitness = self.ga_scheduler.calculate_fitness(ga_solution)
        
        if ga_fitness < best_fitness:
            best_fitness = ga_fitness
            best_solution = copy.deepcopy(ga_solution)
        
        ga_fitness_history = ga_history
        hybrid_fitness_history.extend(ga_history)
        
        # Switch to PSO for exploitation
        print("\nPhase 2: Particle Swarm Exploitation")
        pso_solution, pso_history = self.pso_optimizer.run_pso()
        pso_fitness = self.pso_optimizer.calculate_fitness(pso_solution)
        
        if pso_fitness < best_fitness:
            best_fitness = pso_fitness
            best_solution = copy.deepcopy(pso_solution)
        
        pso_fitness_history = pso_history
        hybrid_fitness_history.extend(pso_history)
        
        # Adaptive refinement phase
        print("\nPhase 3: Adaptive Refinement")
        refined_solution = self.adaptive_refinement(best_solution, ga_solution, pso_solution)
        refined_fitness = self.ga_scheduler.calculate_fitness(refined_solution)
        
        if refined_fitness < best_fitness:
            best_fitness = refined_fitness
            best_solution = refined_solution
        
        print(f"\nHybrid optimization completed:")
        print(f"GA best fitness: {ga_fitness:.2f}")
        print(f"PSO best fitness: {pso_fitness:.2f}")
        print(f"Refined best fitness: {refined_fitness:.2f}")
        print(f"Overall best fitness: {best_fitness:.2f}")
        
        return best_solution, {
            'ga': ga_fitness_history,
            'pso': pso_fitness_history,
            'hybrid': hybrid_fitness_history
        }
    
    def adaptive_refinement(self, best_solution: Dict[int, AGVPlan], 
                          ga_solution: Dict[int, AGVPlan], 
                          pso_solution: Dict[int, AGVPlan]) -> Dict[int, AGVPlan]:
        """Adaptive refinement using local search"""
        print("Performing adaptive refinement...")
        
        refined = copy.deepcopy(best_solution)
        
        # Local search around best solution
        for agv_id, plan in refined.items():
            # Try small improvements
            for _ in range(5):  # 5 attempts per AGV
                # Random small mutation
                if random.random() < 0.3:  # 30% chance
                    # Small schedule adjustment
                    if plan.schedule:
                        adjustment = random.randint(-1, 1)
                        new_start = max(0, plan.schedule[0] + adjustment)
                        plan.schedule = list(range(new_start, new_start + len(plan.path)))
                        plan.total_time = new_start + self.path_planner.calculate_path_travel_time(
                            plan.path, self.agv_lookup[agv_id].max_speed
                        )
        
        return refined

# Initialize hybrid optimizer
hybrid_optimizer = HybridMetaheuristic(nodes, edges, agvs, advanced_planner)

print("Hybrid Metaheuristic initialized")
print(f"Switch frequency: {hybrid_optimizer.switch_frequency} iterations")
print(f"Max iterations: {hybrid_optimizer.max_iterations}")

## Step 5 — Execute all metaheuristics

Run all three approaches and compare their performance:

In [ ]:
def run_metaheuristic_comparison() -> Dict[str, Tuple[Dict[int, AGVPlan], List[float]]]:
    """Run all metaheuristic approaches and compare"""
    results = {}
    
    print("\n" + "="*60)
    print("METAHEURISTIC COMPARISON STUDY")
    print("="*60)
    
    # Run Genetic Algorithm
    print("\n1. RUNNING GENETIC ALGORITHM...")
    ga_start_time = time.time()
    ga_solution, ga_history = ga_scheduler.run_ga()
    ga_time = time.time() - ga_start_time
    ga_fitness = ga_scheduler.calculate_fitness(ga_solution)
    
    results['GA'] = (ga_solution, ga_history, ga_time, ga_fitness)
    
    # Run Particle Swarm Optimization
    print("\n2. RUNNING PARTICLE SWARM OPTIMIZATION...")
    pso_start_time = time.time()
    pso_solution, pso_history = pso_optimizer.run_pso()
    pso_time = time.time() - pso_start_time
    pso_fitness = pso_optimizer.calculate_fitness(pso_solution)
    
    results['PSO'] = (pso_solution, pso_history, pso_time, pso_fitness)
    
    # Run Hybrid Approach
    print("\n3. RUNNING HYBRID APPROACH...")
    hybrid_start_time = time.time()
    hybrid_solution, hybrid_histories = hybrid_optimizer.run_hybrid()
    hybrid_time = time.time() - hybrid_start_time
    hybrid_fitness = ga_scheduler.calculate_fitness(hybrid_solution)
    
    results['Hybrid'] = (hybrid_solution, hybrid_histories['hybrid'], hybrid_time, hybrid_fitness)
    
    # Create comparison table
    comparison_data = []
    for method, (solution, history, comp_time, fitness) in results.items():
        # Calculate solution metrics
        total_time = sum(plan.total_time for plan in solution.values())
        conflicts = ga_scheduler.count_conflicts(solution)
        
        comparison_data.append({
            'Method': method,
            'Computation_Time': comp_time,
            'Fitness_Score': fitness,
            'Total_Completion_Time': total_time,
            'Conflicts': conflicts,
            'AGVs_Scheduled': len(solution),
            'Convergence_Generations': len(history)
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print("\n" + "="*60)
    print("COMPARISON RESULTS")
    print("="*60)
    display(comparison_df)
    
    # Find best method
    best_method = comparison_df.loc[comparison_df['Fitness_Score'].idxmin(), 'Method']
    best_fitness = comparison_df['Fitness_Score'].min()
    
    print(f"\nBest method: {best_method} (Fitness: {best_fitness:.2f})")
    
    return results, comparison_df

# Run the comparison
metaheuristic_results, comparison_df = run_metaheuristic_comparison()

## Step 6 — Comprehensive visualization and analysis

Create detailed visualizations to understand metaheuristic behavior:

In [ ]:
def visualize_metaheuristic_results(results: Dict, comparison_df: pd.DataFrame):
    """Create comprehensive visualization of metaheuristic results"""
    fig, axes = plt.subplots(3, 3, figsize=(20, 16))
    
    # Colors for consistency
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#F7B731']
    
    # ----------------------------
    # Plot 1: Fitness convergence comparison
    # ----------------------------
    ax1 = axes[0, 0]
    
    for i, (method, (solution, history, comp_time, fitness)) in enumerate(results.items()):
        if method == 'Hybrid':
            # Hybrid has combined history
            ax1.plot(history, linewidth=2, label=method, color=colors[i], alpha=0.8)
        else:
            ax1.plot(history, linewidth=2, label=method, color=colors[i], alpha=0.8)
    
    ax1.set_title('Fitness Convergence Comparison')
    ax1.set_xlabel('Iteration/Generation')
    ax1.set_ylabel('Fitness Score (lower is better)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # ----------------------------
    # Plot 2: Computation time comparison
    # ----------------------------
    ax2 = axes[0, 1]
    bars = ax2.bar(comparison_df['Method'], comparison_df['Computation_Time'], 
                   color=colors[:len(comparison_df)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax2.set_title('Computation Time Comparison')
    ax2.set_ylabel('Time (seconds)')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    for bar, value in zip(bars, comparison_df['Computation_Time']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{value:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # ----------------------------
    # Plot 3: Solution quality metrics
    # ----------------------------
    ax3 = axes[0, 2]
    
    x = np.arange(len(comparison_df))
    width = 0.25
    
    ax3.bar(x - width, comparison_df['Fitness_Score'], width, label='Fitness Score', 
            color='#E63946', alpha=0.8)
    ax3.bar(x, comparison_df['Total_Completion_Time']/10, width, label='Completion Time/10', 
            color='#A8DADC', alpha=0.8)
    ax3.bar(x + width, comparison_df['Conflicts']*5, width, label='Conflicts×5', 
            color='#457B9D', alpha=0.8)
    
    ax3.set_title('Solution Quality Metrics')
    ax3.set_ylabel('Score')
    ax3.set_xticks(x)
    ax3.set_xticklabels(comparison_df['Method'], rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # ----------------------------
    # Plot 4-6: Route visualizations for each method
    # ----------------------------
    for i, (method, (solution, history, comp_time, fitness)) in enumerate(results.items()):
        if i < 3:  # Only first 3 methods
            ax = axes[1, i]
            visualize_solution_routes(ax, solution, f'{method} Best Solution')
    
    # ----------------------------
    # Plot 7: AGV utilization analysis
    # ----------------------------
    ax7 = axes[2, 0]
    
    for i, (method, (solution, history, comp_time, fitness)) in enumerate(results.items()):
        if solution:
            completion_times = [plan.total_time for plan in solution.values()]
            agv_ids = list(solution.keys())
            
            ax7.scatter(agv_ids, completion_times, 
                      color=colors[i], alpha=0.7, s=100, label=method)
    
    ax7.set_title('AGV Completion Times by Method')
    ax7.set_xlabel('AGV ID')
    ax7.set_ylabel('Completion Time')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    
    # ----------------------------
    # Plot 8: Performance radar chart
    # ----------------------------
    ax8 = axes[2, 1]
    create_metaheuristic_radar(ax8, comparison_df)
    
    # ----------------------------
    # Plot 9: Statistical summary
    # ----------------------------
    ax9 = axes[2, 2]
    ax9.axis('off')
    
    # Create statistical summary text
    summary_text = "STATISTICAL SUMMARY\n" + "="*25 + "\n\n"
    
    for _, row in comparison_df.iterrows():
        method = row['Method']
        summary_text += f"{method}:\n"
        summary_text += f"  Fitness: {row['Fitness_Score']:.2f}\n"
        summary_text += f"  Time: {row['Computation_Time']:.1f}s\n"
        summary_text += f"  Conflicts: {row['Conflicts']}\n"
        summary_text += f"  Scheduled: {row['AGVs_Scheduled']}/12\n\n"
    
    # Best and worst performers
    best_fitness = comparison_df.loc[comparison_df['Fitness_Score'].idxmin()]
    worst_fitness = comparison_df.loc[comparison_df['Fitness_Score'].idxmax()]
    
    summary_text += f"Best Overall: {best_fitness['Method']}\n"
    summary_text += f"Fastest: {comparison_df.loc[comparison_df['Computation_Time'].idxmin(), 'Method']}\n"
    summary_text += f"Fewest Conflicts: {comparison_df.loc[comparison_df['Conflicts'].idxmin(), 'Method']}"
    
    ax9.text(0.1, 0.9, summary_text, transform=ax9.transAxes, fontsize=10,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def visualize_solution_routes(ax, solution: Dict[int, AGVPlan], title: str):
    """Visualize routes for a specific solution"""
    # Draw terminal layout
    for edge in edges:
        from_node = node_lookup[edge.from_node]
        to_node = node_lookup[edge.to_node]
        ax.plot([from_node.x, to_node.x], [from_node.y, to_node.y], 
               'k-', alpha=0.2, linewidth=1)
    
    # Draw nodes
    for node in nodes:
        ax.scatter(node.x, node.y, s=80, c='lightgray', edgecolors='black')
        ax.text(node.x, node.y, str(node.id), ha='center', va='center', fontsize=8)
    
    # Draw AGV routes (show first 6 to avoid clutter)
    route_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#F7B731', '#5F27CD']
    
    for i, (agv_id, plan) in enumerate(list(solution.items())[:6]):
        if plan.path and len(plan.path) > 1:
            x_coords = [node_lookup[node].x for node in plan.path]
            y_coords = [node_lookup[node].y for node in plan.path]
            
            ax.plot(x_coords, y_coords, 'o-', color=route_colors[i % len(route_colors)], 
                   linewidth=2, markersize=4, alpha=0.8, label=f'AGV {agv_id}')
    
    ax.set_title(title)
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_aspect('equal')

def create_metaheuristic_radar(ax, comparison_df: pd.DataFrame):
    """Create radar chart for metaheuristic comparison"""
    # Normalize metrics for radar chart
    metrics = ['Computation_Time', 'Fitness_Score', 'AGVs_Scheduled']
    
    normalized_data = comparison_df[metrics].copy()
    
    # Inverse time and fitness (lower is better)
    normalized_data['Computation_Time'] = 1 / (normalized_data['Computation_Time'] + 0.001)
    normalized_data['Fitness_Score'] = 1 / normalized_data['Fitness_Score']
    
    # Scale to 0-1 range
    for metric in metrics:
        max_val = normalized_data[metric].max()
        min_val = normalized_data[metric].min()
        if max_val > min_val:
            normalized_data[metric] = (normalized_data[metric] - min_val) / (max_val - min_val)
        else:
            normalized_data[metric] = 1.0
    
    # Create radar chart
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]
    
    ax.clear()
    ax = plt.subplot(111, projection='polar')
    
    radar_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    for i, (idx, row) in enumerate(normalized_data.iterrows()):
        values = row.tolist()
        values += values[:1]
        
        ax.plot(angles, values, 'o-', linewidth=2, label=comparison_df.loc[idx, 'Method'],
               color=radar_colors[i % len(radar_colors)])
        ax.fill(angles, values, alpha=0.25, color=radar_colors[i % len(radar_colors)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['Speed', 'Quality', 'Coverage'])
    ax.set_ylim(0, 1)
    ax.set_title('Metaheuristic Performance Radar')
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# Create visualizations
visualize_metaheuristic_results(metaheuristic_results, comparison_df)

## Step 7 — Scalability and performance analysis

Test how metaheuristics perform on different problem sizes:

In [ ]:
def metaheuristic_scalability_test():
    """Test scalability of metaheuristics with different problem sizes"""
    print("=== METAHEURISTIC SCALABILITY ANALYSIS ===")
    
    # Test different problem sizes
    test_sizes = [6, 8, 10, 12]  # Number of AGVs
    scalability_results = []
    
    for num_agvs in test_sizes:
        print(f"\nTesting with {num_agvs} AGVs...")
        
        # Create test instance (subset of AGVs)
        test_agvs = agvs[:num_agvs]
        
        # Create reduced-size optimizers
        test_ga = GeneticAlgorithmScheduler(nodes, edges, test_agvs, advanced_planner)
        test_ga.population_size = 30  # Reduced for speed
        test_ga.generations = 50     # Reduced for speed
        
        test_pso = ParticleSwarmOptimizer(nodes, edges, test_agvs, advanced_planner)
        test_pso.swarm_size = 20     # Reduced for speed
        test_pso.max_iterations = 40  # Reduced for speed
        
        # Test GA
        start_time = time.time()
        ga_solution, ga_history = test_ga.run_ga()
        ga_time = time.time() - start_time
        ga_fitness = test_ga.calculate_fitness(ga_solution)
        
        # Test PSO
        start_time = time.time()
        pso_solution, pso_history = test_pso.run_pso()
        pso_time = time.time() - start_time
        pso_fitness = test_pso.calculate_fitness(pso_solution)
        
        scalability_results.append({
            'Num_AGVs': num_agvs,
            'GA_Time': ga_time,
            'GA_Fitness': ga_fitness,
            'PSO_Time': pso_time,
            'PSO_Fitness': pso_fitness,
            'GA_Scheduled': len(ga_solution),
            'PSO_Scheduled': len(pso_solution)
        })
        
        print(f"  GA: {ga_time:.1f}s, fitness: {ga_fitness:.2f}, scheduled: {len(ga_solution)}")
        print(f"  PSO: {pso_time:.1f}s, fitness: {pso_fitness:.2f}, scheduled: {len(pso_solution)}")
    
    # Create scalability DataFrame
    scalability_df = pd.DataFrame(scalability_results)
    display(scalability_df)
    
    # Visualize scalability
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Computation time scalability
    ax1.plot(scalability_df['Num_AGVs'], scalability_df['GA_Time'], 
            'o-', linewidth=2, markersize=8, color='#FF6B6B', label='Genetic Algorithm')
    ax1.plot(scalability_df['Num_AGVs'], scalability_df['PSO_Time'], 
            's-', linewidth=2, markersize=8, color='#4ECDC4', label='Particle Swarm')
    
    ax1.set_title('Computation Time Scalability')
    ax1.set_xlabel('Number of AGVs')
    ax1.set_ylabel('Computation Time (seconds)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Solution quality scalability
    ax2.plot(scalability_df['Num_AGVs'], scalability_df['GA_Fitness'], 
            'o-', linewidth=2, markersize=8, color='#FF6B6B', label='Genetic Algorithm')
    ax2.plot(scalability_df['Num_AGVs'], scalability_df['PSO_Fitness'], 
            's-', linewidth=2, markersize=8, color='#4ECDC4', label='Particle Swarm')
    
    ax2.set_title('Solution Quality Scalability')
    ax2.set_xlabel('Number of AGVs')
    ax2.set_ylabel('Fitness Score (lower is better)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Success rate scalability
    ax3.plot(scalability_df['Num_AGVs'], scalability_df['GA_Scheduled'], 
            'o-', linewidth=2, markersize=8, color='#FF6B6B', label='Genetic Algorithm')
    ax3.plot(scalability_df['Num_AGVs'], scalability_df['PSO_Scheduled'], 
            's-', linewidth=2, markersize=8, color='#4ECDC4', label='Particle Swarm')
    
    ax3.set_title('Scheduling Success Rate')
    ax3.set_xlabel('Number of AGVs')
    ax3.set_ylabel('Number of AGVs Successfully Scheduled')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return scalability_df

# Run scalability analysis
scalability_results = metaheuristic_scalability_test()

## Summary and key insights

This Tier-3 notebook demonstrates **metaheuristic approaches** for AGV traffic management:

### Metaheuristic strengths
- **Global search**: Avoid local optima through population-based exploration
- **Balance**: Trade-off between solution quality and computation time
- **Flexibility**: Adaptable to different problem sizes and constraints
- **Robustness**: Handle complex, non-linear optimization landscapes

### Algorithm comparison insights
- **Genetic Algorithm**: Good exploration, diverse solutions, moderate speed
- **Particle Swarm**: Fast convergence, good exploitation, requires tuning
- **Hybrid approach**: Best of both worlds, higher computational cost

### Performance characteristics
- **Convergence behavior**: Different patterns for each algorithm
- **Solution quality**: Consistent improvement over heuristics
- **Scalability**: Handle larger instances than exact methods
- **Computation time**: Higher than heuristics but reasonable for medium instances

### Practical considerations
- **Parameter tuning**: Critical for metaheuristic performance
- **Problem representation**: Encoding affects solution quality
- **Hybridization**: Combining methods often yields best results
- **Real-world applicability**: Suitable for offline planning and near-real-time

### Next steps
- Investigate reinforcement learning for dynamic adaptation in Tier-4
- Explore parallelization for larger scale problems
- Consider multi-objective optimization (time, energy, fairness)
- Study integration with real terminal management systems